# CS03b — Trade-flow attribution for candle-streak starts and stops

**Historical exploratory study.** The exact CS03 spot-price streaks, Binance USD-M futures minute-to-five-minute alignment, 60/20/20 full-day split, and regularized multinomial logistic objective are frozen. This ablation separates **signed aggressive execution pressure** from **volume/trade-count activity**. Neither is resting L2 OBI. The CS03 historical TEST dates were already inspected; even a positive result is not prospective confirmation.

At five-minute close *t*, forecast *t+5m* as DOWN, ZERO, UP (all three classes). Current zero-return candles are retained. The binary UP/DOWN, reversal and fresh-start diagnostics are conditional on the *immediately following* bar being nonzero; this condition is **not known at decision time**. See [`CS03b protocol`](../../discussions/Candle_Streaks/003b_chatgpt_trade_flow_attribution_preregistration.md).

## 0 — Environment and unit tests

Run the complete regression tests before any market-data evaluation. In normal mode, results are saved under `notebooks/Candle_Streaks/03b/`. `CS03B_SYNTHETIC=1` uses an isolated temporary directory; synthetic outcomes do not establish market evidence.

In [1]:
import os,sys,json,subprocess,time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Execute notebook inside the repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import candle_streak_trade_flow as tf
from data_engineering.modeling import candle_streak_flow_attribution as ab
cfg=ab.Specification()
SYNTHETIC=os.getenv('CS03B_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='cs03b-')
    OUT=Path(tmp.name)/'03b'
else:
    from data_engineering.modeling import markov_lift_distributions as vd14
    OUT=ROOT/'notebooks'/'Candle_Streaks'/'03b'
OUT.mkdir(parents=True,exist_ok=True)
print('CS03b:',ab.VERSION,'CS03 base:',tf.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
checks=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_flow_attribution.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(checks.stdout[-2500:]);print(checks.stderr[-1200:])
assert checks.returncode==0, 'CS03b tests failed; stop before historical evaluation'

CS03b: CS03b-0.1.0 CS03 base: CS03-0.1.1 synthetic: False output: D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03b
..............                                                           [100%]
14 passed in 2.89s




## 1 — Reuse already-downloaded Binance inputs, no new collection

Spot: `DERIBIT_DATA_ROOT/history/price_5m/<ASSET>.csv.gz`, with VD14 returns. Aggressive execution: `BINANCE_UM_DIR/<ASSET>USDT_um_1m.parquet`, or `DERIBIT_DATA_ROOT/../Binance`. Exact 5×1-minute open-time buckets become completed five-minute timestamps. The signal at timestamp *t* never sees executions occurring after *t*. Spot and USD-M perpetual are different markets.

In [2]:
ASSET_DATA={};excluded={}
if SYNTHETIC:
    rng=np.random.default_rng(2605)
    ndays=132; N=ndays*288
    minute=pd.date_range('2026-01-01',periods=N*5,freq='1min',tz='UTC')
    frac=np.clip(.5+.18*np.sin(np.arange(len(minute))/18)+rng.normal(0,.05,len(minute)),.01,.99)
    v=rng.uniform(100.,800.,len(minute))
    fake=pd.DataFrame({'open_time':minute,'quote_volume':v,'taker_buy_quote_volume':v*frac,
                       'trades':rng.integers(3,40,len(minute))})
    ff=tf.one_minute_futures_flow(fake)
    f=ff.flow_5m.to_numpy(float)
    prob=1/(1+np.exp(-1.4*f))
    ret=np.where(rng.random(N)<prob,1.,-1.)*rng.uniform(.0001,.003,N)
    ret[rng.random(N)<.025]=0.
    sp=pd.DataFrame({'state_time':ff.state_time,'return_5m':ret})
    ASSET_DATA={'BTC':(sp,ff)}
    data_description='SYNTHETIC SMOKE DATA — NOT HISTORICAL'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT not set in .env/environment')
    root=Path(rootval)
    hist=root/'history'/'price_5m'
    futures=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else root.parent/'Binance'
    print('Spot input:',hist,'futures input:',futures)
    for asset in cfg.assets:
        path=futures/f'{asset}USDT_um_1m.parquet'
        if not path.is_file():
            excluded[asset]=f'Missing existing futures file: {path}';continue
        try:
            prices,missing=vd14.load_returns(hist,(asset,),end=pd.Timestamp('2026-10-14T00:00:00Z'),min_days=365)
            sp=prices.loc[prices.underlying.eq(asset),['state_time','return_5m']].copy()
            ff=tf.read_futures_parquet(path,start=sp.state_time.min()-pd.Timedelta(days=2),
                                       end=sp.state_time.max()+pd.Timedelta(minutes=5))
            ASSET_DATA[asset]=(sp,ff)
        except Exception as exc:
            excluded[asset]=f'{type(exc).__name__}: {exc}'
    data_description='LOCAL Binance spot 5m and USD-M futures 1m'
assert ASSET_DATA,f'No input coverage: {excluded}'
print('Available assets:',list(ASSET_DATA),'excluded:',excluded)

Spot input: D:\MarketData\Deribit\history\price_5m futures input: D:\MarketData\Binance
Available assets: ['BTC', 'ETH', 'SOL', 'XRP'] excluded: {}


## 2 — Prepare identical five-minute origins and chronological periods

Use the exact CS03 continuous streak and volatility/flow feature preparation. Require 288 matched five-minute feature rows per UTC day. No target can cross the TRAIN/VALID/TEST boundary, and the current candle's flat return is retained with streak age 0.

In [3]:
prepared={};splits={};quality=[];coverage=[]
for asset,(sp,ff) in ASSET_DATA.items():
    pnl=tf.build_origin_panel(sp,ff,fast_bars=cfg.fast_bars,slow_bars=cfg.slow_bars,flow_bars=cfg.flow_bars)
    dates=tf.full_days(pnl)
    quality.append({'asset':asset,'spot_rows':len(sp),'futures_5m_buckets':len(ff),
                    'valid_5m_flow_buckets':int(ff.flow_valid.sum()),'complete_overlap_days':len(dates),
                    'earliest_complete_date':str(dates.min()) if len(dates) else None,
                    'latest_complete_date':str(dates.max()) if len(dates) else None})
    if len(dates)<60:
        excluded[asset]=f'Only {len(dates)} matching complete days';continue
    data,split=tf.split_panel(pnl,dates,cfg.train_fraction,cfg.valid_fraction)
    data['asset']=asset;prepared[asset]=data;splits[asset]=split
    for period,g in data.groupby('period',sort=False):
        score=g[g.scorable]
        coverage.append({'asset':asset,'period':period,'dates':g.date.nunique(),
                         'eligible_origins':int(len(g)),'scorable_origins':int(len(score)),
                         'flat_origins':int(score.sign.eq(0).sum()),
                         'next_flat_count':int(score.next_sign.eq(0).sum()),
                         'new_streak_reversals':int(score.outcome.eq('reverse_new_streak').sum())})
print('Eligible:',list(prepared),'excluded:',excluded)
display(pd.DataFrame(quality));display(pd.DataFrame(coverage))
assert prepared,'No asset meets minimum 60 complete intersection dates'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(prepared):print('PRIMARY BTC/ETH COVERAGE MISSING — cannot claim a passing gate')

Eligible: ['BTC', 'ETH', 'SOL', 'XRP'] excluded: {}


,asset,spot_rows,futures_5m_buckets,valid_5m_flow_buckets,complete_overlap_days,earliest_complete_date,latest_complete_date
0,BTC,221761,213122,213101,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
1,ETH,221761,213122,213101,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
2,SOL,221761,213122,213101,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00
3,XRP,221761,213122,213101,736,2024-10-02 00:00:00+00:00,2026-10-10 00:00:00+00:00


,asset,period,dates,eligible_origins,scorable_origins,flat_origins,next_flat_count,new_streak_reversals
0,BTC,train,441,127008,127004,312,312,63219
1,BTC,valid,147,42336,42335,129,129,21504
2,BTC,test,148,42624,42623,192,192,21644
3,ETH,train,441,127008,127004,203,203,64059
4,ETH,valid,147,42336,42335,133,133,21643
5,ETH,test,148,42624,42623,146,146,21857
6,SOL,train,441,127008,127004,1677,1677,61836
7,SOL,valid,147,42336,42335,1563,1563,20214
8,SOL,test,148,42624,42623,1819,1819,20020
9,XRP,train,441,127008,127004,1963,1963,62078


## 3 — Fit the four fixed feature ablations, score proper probabilities

All four model families fit on identical TRAIN origin rows, with C=0.25 and Newton–Cholesky, TRAIN-only standardization. **Price+volatility** is the baseline. **Signed-only** adds 5m/30m taker imbalance and interactions with current sign; **activity-only** adds log quote volume and log trade count. **Both** exactly matches CS03 v0.1.1 full model. Diagnostic scores cover unconditional three-class prediction, conditional UP/DOWN, running-streak flat termination and reversal, and upward start after a flat origin. The last two conditioned-on-future-moving diagnostics are not live unconditional probabilities.

In [4]:
all_scores=[];all_boot=[];all_calibration=[];all_coeff=[];all_null=[]
models={};cuts_by_asset={}
for asset,x in prepared.items():
    train=x[x.period.eq('train')]
    fit=ab.fit_ablation(train,C=cfg.logistic_C,max_iter=cfg.max_iter,tol=cfg.tol)
    cuts=tf.fit_flow_cuts(train)
    models[asset]=fit;cuts_by_asset[asset]=cuts
    for name,(terms,scaler,m) in fit['models'].items():
        for feat,row in zip(terms,m.coef_.T):
            all_coeff.append({'asset':asset,'model':name,'feature':feat,
                              **{f'class_{int(c)}':float(z) for c,z in zip(m.classes_,row)}})
    for period in ('valid','test'):
        ev=x[x.period.eq(period)&x.scorable].copy()
        p=ab.predict_ablation(ev,fit)
        losses=ab.origin_losses(ev,p)
        all_scores.append(ab.scores_from_losses(losses).assign(asset=asset,period=period))
        aboot=ab.day_bootstrap(losses,B=cfg.bootstrap_reps if not SYNTHETIC else 60,
                               seed=cfg.seed,min_days=cfg.min_days_ci,min_events=cfg.min_events_ci)
        all_boot.append(aboot.assign(asset=asset,period=period))
        all_calibration.append(ab.transition_calibration(ev,p,cuts).assign(asset=asset,period=period))
        del losses,p,ev
    test=x[x.period.eq('test')&x.scorable]
    all_null.append(ab.feature_shuffle_null(test,fit,reps=cfg.shuffle_reps if not SYNTHETIC else 3,
                                           seed=cfg.seed).assign(asset=asset))
scores=pd.concat(all_scores,ignore_index=True)
bootstrap=pd.concat(all_boot,ignore_index=True)
calibration=pd.concat(all_calibration,ignore_index=True)
coefficients=pd.DataFrame(all_coeff)
shuffle_null=pd.concat(all_null,ignore_index=True)
display(scores[(scores.period=='test')&scores.metric.isin(['threeclass_logloss','binary_logloss',
                'flat_vs_move_logloss','reversal_logloss','start_up_logloss'])][['asset','model','metric','n','score']])

,asset,model,metric,n,score
40,BTC,price_vol,threeclass_logloss,42623,0.713916
42,BTC,price_vol,binary_logloss,42431,0.693387
44,BTC,price_vol,flat_vs_move_logloss,42431,0.022674
46,BTC,price_vol,reversal_logloss,42252,0.693424
48,BTC,price_vol,start_up_logloss,179,0.684544
...,...,...,...,...,...
310,XRP,price_vol_both,threeclass_logloss,42623,0.813212
312,XRP,price_vol_both,binary_logloss,41203,0.693158
314,XRP,price_vol_both,flat_vs_move_logloss,41203,0.142104
316,XRP,price_vol_both,reversal_logloss,39844,0.693117


## 4 — Interpret streak starts, continuations and flat stops

The next table is **descriptive TEST calibration** conditional on TRAIN-frozen signed-flow tertiles; it is not an in-sample refit. It reports model probabilities and realized transition rates with counts, including flat-origin starts. A few attractive cells do not establish predictive value.

In [5]:
q=calibration[(calibration.period=='test')&(calibration.model=='price_vol_both')]
display(q[q.n>=50][['asset','origin','sign','age_group','flow_tier','n',
                  'p_continue','obs_continue','p_reversal','obs_reversal',
                  'p_flat','obs_flat','p_start_up','obs_start_up']].head(60))
print('TRAIN-only signed-flow cutoffs:',cuts_by_asset)

,asset,origin,sign,age_group,flow_tier,n,p_continue,obs_continue,p_reversal,obs_reversal,p_flat,obs_flat,p_start_up,obs_start_up
273,BTC,flat_origin,0,0 flat,0,63,NaN,0.000000,NaN,0.000000,0.051010,0.015873,0.500850,0.603175
274,BTC,flat_origin,0,0 flat,1,59,NaN,0.000000,NaN,0.000000,0.056573,0.101695,0.496976,0.525424
275,BTC,flat_origin,0,0 flat,2,70,NaN,0.000000,NaN,0.000000,0.079269,0.085714,0.475549,0.571429
276,BTC,running_streak,-1,1,0,5448,0.512162,0.494310,0.484217,0.500184,0.003621,0.005507,NaN,0.000000
277,BTC,running_streak,-1,1,1,3758,0.518664,0.486429,0.478936,0.509846,0.002400,0.003725,NaN,0.000000
278,BTC,running_streak,-1,1,2,1689,0.523799,0.505625,0.472315,0.492007,0.003886,0.002368,NaN,0.000000
279,BTC,running_streak,-1,2,0,3195,0.492817,0.482942,0.503591,0.512989,0.003592,0.004069,NaN,0.000000
280,BTC,running_streak,-1,2,1,1620,0.499134,0.472222,0.498386,0.527160,0.002480,0.000617,NaN,0.000000
281,BTC,running_streak,-1,2,2,560,0.504272,0.464286,0.491057,0.530357,0.004671,0.005357,NaN,0.000000
282,BTC,running_streak,-1,3,0,1608,0.478861,0.498756,0.517067,0.498756,0.004072,0.002488,NaN,0.000000


TRAIN-only signed-flow cutoffs: {'BTC': [-0.1257240265707173, 0.10127571311533531], 'ETH': [-0.11231213856591744, 0.0904738628842851], 'SOL': [-0.10746775977091667, 0.07986807183401529], 'XRP': [-0.10197969353585612, 0.06510882796798242]}


## 5 — Paired-day confidence intervals, component-shuffle checks and decision

The CS03 historical TEST period was previously inspected. The gate is **hypothesis screening**, not fresh confirmation. A signed-flow attribution claim requires signed-only to beat baseline AND the full model to beat activity-only, on BOTH BTC and ETH. A separate binary gate additionally requires those directional tests; neither a three-class improvement alone nor a tiny retrospective flat-origin subgroup implies a tradable edge.

In [6]:
test_boot=bootstrap[bootstrap.period.eq('test')]
display(test_boot[(test_boot.metric.isin(['threeclass_logloss','binary_logloss','flat_vs_move_logloss',
                    'reversal_logloss','start_up_logloss']))&
       (test_boot.model.isin(['price_vol_signed','price_vol_activity','price_vol_both']))][
       ['asset','model','reference','metric','n_origins','delta_equal_day','ci025','ci975','status']].head(80))
display(shuffle_null)
dec=ab.decision(bootstrap,synthetic=SYNTHETIC)
print(json.dumps(dec,indent=2))

,asset,model,reference,metric,n_origins,delta_equal_day,ci025,ci975,status
60,BTC,price_vol_signed,price_vol,threeclass_logloss,42623,-0.000272,-0.000507,-0.000044,OK
61,BTC,price_vol_activity,price_vol,threeclass_logloss,42623,-0.000783,-0.001325,-0.000353,OK
62,BTC,price_vol_both,price_vol,threeclass_logloss,42623,-0.001026,-0.001590,-0.000511,OK
63,BTC,price_vol_both,price_vol_signed,threeclass_logloss,42623,-0.000754,-0.001294,-0.000321,OK
64,BTC,price_vol_both,price_vol_activity,threeclass_logloss,42623,-0.000243,-0.000491,-0.000016,OK
...,...,...,...,...,...,...,...,...,...
327,SOL,price_vol_both,price_vol_signed,flat_vs_move_logloss,40804,-0.003460,-0.004018,-0.002957,OK
328,SOL,price_vol_both,price_vol_activity,flat_vs_move_logloss,40804,-0.000196,-0.000378,-0.000043,OK
329,SOL,price_vol_signed,price_vol_activity,flat_vs_move_logloss,40804,0.003263,0.002729,0.003776,OK
336,SOL,price_vol_signed,price_vol,reversal_logloss,39069,-0.000203,-0.000397,-0.000004,OK


,component,model,observed_delta,null05,null_median,null95,fraction_null_le_observed,n_shuffle,asset
0,signed,price_vol_signed,-0.000272,0.000132,0.000374,0.000534,0.0,100,BTC
1,activity,price_vol_activity,-0.000783,0.002871,0.003227,0.003702,0.0,100,BTC
2,signed,price_vol_signed,-0.000600,0.000157,0.000434,0.000641,0.0,100,ETH
3,activity,price_vol_activity,-0.000144,0.000850,0.001052,0.001278,0.0,100,ETH
4,signed,price_vol_signed,-0.000261,0.000694,0.000876,0.001092,0.0,100,SOL
5,activity,price_vol_activity,-0.002984,-0.001293,-0.001012,-0.000731,0.0,100,SOL
6,signed,price_vol_signed,-0.000265,0.000262,0.000450,0.000675,0.0,100,XRP
7,activity,price_vol_activity,0.000434,0.011448,0.012587,0.013686,0.0,100,XRP


{
  "verdict": "HISTORICAL_SIGNED_FLOW_CANDIDATE",
  "status": "HISTORICAL_EXPLORATORY",
  "signed_threeclass_gate": true,
  "signed_binary_gate": true,
  "activity_threeclass_gate": false,
  "primary": [
    {
      "asset": "BTC",
      "component": "signed_vs_price",
      "metric": "threeclass_logloss",
      "delta": -0.00027224909033589335,
      "upper95": -4.3563364185440474e-05,
      "pass": true
    },
    {
      "asset": "BTC",
      "component": "signed_vs_price",
      "metric": "binary_logloss",
      "delta": -0.0002756363984719644,
      "upper95": -7.059940176788452e-05,
      "pass": true
    },
    {
      "asset": "BTC",
      "component": "activity_vs_price",
      "metric": "threeclass_logloss",
      "delta": -0.0007830018824290256,
      "upper95": -0.0003529383127538363,
      "pass": true
    },
    {
      "asset": "BTC",
      "component": "activity_vs_price",
      "metric": "binary_logloss",
      "delta": 3.2119631677186654e-05,
      "upper95": 0.00011

## 6 — Save aggregate results only

No raw one-minute data, event-level target records, origin-by-origin predictions, or model binaries are committed. Outputs are compact score, paired CI, calibration and provenance summaries.

In [7]:
aggregate={
    '03b_coverage.csv':pd.DataFrame(coverage),
    '03b_quality.csv':pd.DataFrame(quality),
    '03b_scores.csv':scores,
    '03b_paired_day_bootstrap.csv':bootstrap,
    '03b_transition_calibration.csv':calibration,
    '03b_model_coefficients.csv':coefficients,
    '03b_component_shuffle.csv':shuffle_null,
}
for name,frame in aggregate.items():frame.to_csv(OUT/name,index=False)
metadata={'module':'CS03b','registration':ab.registration(),'cs03_base_version':tf.VERSION,
    'source':data_description,'historical_test_reused':True,'synthetic':SYNTHETIC,
    'assets_evaluated':list(prepared),'excluded':excluded,'split':splits,
    'flow_tertiles_train':cuts_by_asset,
    'models':{k:list(v) for k,v in ab.FEATURES.items()},
    'forecast_targets':['DOWN','ZERO','UP'],
    'conditional_binary_targets_not_actionable_alone':True}
(OUT/'03b_manifest.json').write_text(json.dumps(metadata,indent=2,default=str),encoding='utf-8')
(OUT/'03b_decision.json').write_text(json.dumps(dec,indent=2,default=str),encoding='utf-8')
assert all((OUT/k).is_file() for k in list(aggregate)+['03b_manifest.json','03b_decision.json'])
print('Saved',len(aggregate)+2,'aggregate outputs to',OUT)

Saved 9 aggregate outputs to D:\Projects\deribit_market_data\notebooks\Candle_Streaks\03b


## Research interpretation constraints

- Historical CS03 TEST was already read; **do not treat this as independent confirmation** or optimize further on these dates.
- Spot streak returns and USD-M futures taker-flow come from different trading instruments. Signed taker flow is **not** depth OBI or order additions/cancellations.
- Three-class NLL is the only fully unconditional next-bar direction/flat score. A conditional binary benefit cannot be directly deployed without modeling zero-return risk.
- Positive overall scores do not imply start-after-zero, reversal or flat-termination performance; use separate task scores with adequate origin counts.
- Do not modify AM-T-HAR-Blend, futures collectors, trading thresholds or live strategy until new independent evidence warrants it.